# Logistic Regression Workflow

**Project question:** How do I fit and interpret a binary-response regression model?

By the end of this notebook, you should be able to:

- fit a logistic model with explicit categorical reference levels
- translate log-odds coefficients into odds ratios with confidence intervals
- produce conditional predicted probabilities without causal overclaiming

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [1]:

from lite_setup import ensure_packages
await ensure_packages()

Using the current Python environment.


In [2]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [3]:
import statsmodels.formula.api as smf

In [4]:
df = pd.read_csv(DATA / 'simulated_churn.csv')
formula = (
    'churn ~ tenure_months + monthly_charge + support_tickets + usage_gb '
    '+ C(contract, Treatment(reference="month_to_month")) '
    '+ C(autopay, Treatment(reference="no")) + satisfaction'
)
model = smf.logit(formula, data=df).fit(disp=False)
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                  churn   No. Observations:                  320
Model:                          Logit   Df Residuals:                      311
Method:                           MLE   Df Model:                            8
Date:                Thu, 06 Aug 2026   Pseudo R-squ.:                  0.2887
Time:                        00:25:30   Log-Likelihood:                -147.81
converged:                       True   LL-Null:                       -207.80
Covariance Type:            nonrobust   LLR p-value:                 3.358e-22
==================================================================================================================================
                                                                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------------------
Intercept                                                          5.0262      1.070      4.695      0.000       2.928       7.124
C(contract, Treatment(reference="month_to_month"))[T.one_year]    -1.4169      0.352     -4.025      0.000      -2.107      -0.727
C(contract, Treatment(reference="month_to_month"))[T.two_year]    -1.7238      0.538     -3.205      0.001      -2.778      -0.670
C(autopay, Treatment(reference="no"))[T.yes]                      -0.7552      0.302     -2.497      0.013      -1.348      -0.162
tenure_months                                                     -0.0229      0.008     -2.911      0.004      -0.038      -0.007
monthly_charge                                                     0.0245      0.008      2.942      0.003       0.008       0.041
support_tickets                                                    0.4247      0.213      1.992      0.046       0.007       0.843
usage_gb                                                          -0.0296      0.006     -5.342      0.000      -0.040      -0.019
satisfaction                                                      -0.5259      0.095     -5.530      0.000      -0.712      -0.340
==================================================================================================================================
"""

The categorical coefficients compare with `month_to_month` for contract and `no` for autopay. Maximum likelihood estimates the coefficients; the reported standard errors and intervals rely on the fitted-model assumptions.

In [5]:
interval = model.conf_int()
coef = model.params.to_frame('log_odds_coef')
coef['odds_ratio'] = np.exp(coef['log_odds_coef'])
coef['odds_ratio_ci_low'] = np.exp(interval[0])
coef['odds_ratio_ci_high'] = np.exp(interval[1])
coef

,log_odds_coef,odds_ratio,odds_ratio_ci_low,odds_ratio_ci_high
Intercept,5.026244,152.359606,18.692651,1241.848952
"C(contract, Treatment(reference=""month_to_month""))[T.one_year]",-1.416904,0.242464,0.121613,0.483407
"C(contract, Treatment(reference=""month_to_month""))[T.two_year]",-1.723808,0.178385,0.062161,0.511916
"C(autopay, Treatment(reference=""no""))[T.yes]",-0.755187,0.469923,0.259740,0.850184
tenure_months,-0.022906,0.977354,0.962399,0.992542
monthly_charge,0.024481,1.024783,1.008208,1.041630
support_tickets,0.424662,1.529074,1.006794,2.322289
usage_gb,-0.029585,0.970848,0.960367,0.981444
satisfaction,-0.525916,0.591014,0.490502,0.712123


In [6]:
ticket_or = coef.loc['support_tickets']
print(
    'Holding the listed predictors constant, one additional support ticket '
    f"multiplies the estimated churn odds by {ticket_or['odds_ratio']:.2f} "
    f"(95% CI {ticket_or['odds_ratio_ci_low']:.2f} to {ticket_or['odds_ratio_ci_high']:.2f})."
)

Holding the listed predictors constant, one additional support ticket multiplies the estimated churn odds by 1.53 (95% CI 1.01 to 2.32).


An odds ratio is multiplicative on odds, not an absolute probability increase. The probability change depends on all predictors and the starting probability.

In [7]:
new_customers = pd.DataFrame({
    'tenure_months': [6, 36],
    'monthly_charge': [85, 60],
    'support_tickets': [3, 0],
    'usage_gb': [60, 110],
    'contract': ['month_to_month', 'two_year'],
    'autopay': ['no', 'yes'],
    'satisfaction': [5.0, 8.5]
})
new_customers['predicted_churn_probability'] = model.predict(new_customers)
new_customers

,tenure_months,monthly_charge,support_tickets,usage_gb,contract,autopay,satisfaction,predicted_churn_probability
0,6,85,3,60,month_to_month,no,5.0,0.978937
1,36,60,0,110,two_year,yes,8.5,0.010632


**Interpretation:** These predictions are conditional on the specified profiles and the synthetic data-generating relationship. They are not treatment effects: changing a support-ticket count in the table does not prove that causing another ticket would change churn.